# Notebook 08 — RQ3: Four-Quadrant Consistency Analysis ([#117](https://github.com/UBC-MDS/UBC-MDS-Soccer-Capstone-2026/issues/117))

**Owner:** Rabin (W4)  
**Research question (RQ3):** Do players maintain similar tactical performance profiles between club and national contexts?

**Inputs (Databricks → BigQuery foreign catalog):**
- `dbt_intermediate.int_player_club_vs_national` — 13 PCA-aligned context z-scores for dual-context players
- `analytics.pca_loadings` — PCA feature weights from `src/ml/cluster.py`
- `analytics.consistency_scores` (optional) — production output from `src/ml/consistency.py`
- `dbt_intermediate.int_player_match_stats` + `matches` — position and competition context

**Deliverables in this notebook:**
1. End-to-end methodology (feature set, weighting, score formulas, quadrant thresholding)
2. Quadrant distribution table
3. Top 10 notable players per quadrant (club + national score)
4. Position breakdown (including defenders vs forwards)
5. Competition breakdown (La Liga, EPL, Bundesliga, Serie A, Ligue 1)
6. Scatter plot of `club_performance_score` vs `national_performance_score` colored by quadrant

---
## 0. Setup

In [ ]:
import os
import warnings

warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import mannwhitneyu

pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.3f}".format)
sns.set_theme(style="whitegrid", palette="muted")

BQ_CATALOG = os.environ.get("BQ_CATALOG", "bq_raw_statsbomb_sa_catalog")
INTERMEDIATE = f"{BQ_CATALOG}.dbt_intermediate"
INT_TABLE = f"{INTERMEDIATE}.int_player_club_vs_national"
PCA_LOADINGS = f"{BQ_CATALOG}.analytics.pca_loadings"
CONSISTENCY_TABLE = f"{BQ_CATALOG}.analytics.consistency_scores"
INT_MATCH = f"{INTERMEDIATE}.int_player_match_stats"
MATCHES = f"{BQ_CATALOG}.raw_statsbomb.matches"
MIN_MINUTES = 270
USE_PRECOMPUTED_SCORES = False  # True = read analytics.consistency_scores instead of recomputing

TOP5_RAW_NAMES = [
    "La Liga",
    "Premier League",
    "English Premier League",
    "1. Bundesliga",
    "Bundesliga",
    "Serie A",
    "Ligue 1",
]
NAME_TO_LABEL = {
    "La Liga": "La Liga",
    "Premier League": "EPL",
    "English Premier League": "EPL",
    "1. Bundesliga": "Bundesliga",
    "Bundesliga": "Bundesliga",
    "Serie A": "Serie A",
    "Ligue 1": "Ligue 1",
}
COMP_LABEL_ORDER = ["La Liga", "EPL", "Bundesliga", "Serie A", "Ligue 1"]

# Keep in sync with src/ml/consistency.py and src/ml/cluster.py (13 features).
FEATURES = [
    "xg_per_90",
    "shots_per_90",
    "passes_per_90",
    "passes_att_third_per_90",
    "pressures_per_90",
    "carries_per_90",
    "dribbles_per_90",
    "interceptions_per_90",
    "blocks_per_90",
    "clearances_per_90",
    "duels_per_90",
    "xg_per_shot",
    "pass_completion_pct",
]
Z_COLS = [f"z_{f}" for f in FEATURES]

POSITION_MAP = {
    "Goalkeeper": "GK",
    "Center Back": "CB",
    "Left Center Back": "CB",
    "Right Center Back": "CB",
    "Left Back": "FB",
    "Right Back": "FB",
    "Left Wing Back": "FB",
    "Right Wing Back": "FB",
    "Center Defensive Midfield": "CM",
    "Left Defensive Midfield": "CM",
    "Right Defensive Midfield": "CM",
    "Center Midfield": "CM",
    "Left Center Midfield": "CM",
    "Right Center Midfield": "CM",
    "Left Midfield": "CM",
    "Right Midfield": "CM",
    "Attacking Midfield": "AM",
    "Center Attacking Midfield": "AM",
    "Left Attacking Midfield": "AM",
    "Right Attacking Midfield": "AM",
    "Left Wing": "AM",
    "Right Wing": "AM",
    "Secondary Striker": "AM",
    "Center Forward": "FW",
    "Striker": "FW",
    "Left Center Forward": "FW",
    "Right Center Forward": "FW",
}
POSITION_ORDER = ["GK", "CB", "FB", "CM", "AM", "FW"]

print(f"INT table:     {INT_TABLE}")
print(f"PCA loadings:  {PCA_LOADINGS}")
print(f"Scores table:  {CONSISTENCY_TABLE} (optional)")
print("Setup complete.")

---
## 1. Methodology (documented)

### 1.1 Feature selection
- Use the same **13** per-90 features as `src/ml/consistency.py` and `src/ml/cluster.py` (PR #124).
- Input z-scores come from `dbt_intermediate.int_player_club_vs_national`, normalized within context (`is_international`).

### 1.2 PCA weighting
- For each feature `f`, compute weight:
  \( w_f = \frac{\sum_i |loading_{i,f}|}{\sum_{f'}\sum_i |loading_{i,f'}|} \)
- Feature names in `analytics.pca_loadings` use long names (`xg_per_90`, `passes_per_90`, …).

### 1.3 Context performance scores
- `club_performance_score` = weighted sum of club-context z-features.
- `national_performance_score` = weighted sum of national-context z-features.

### 1.4 Consistency score
- \( consistency = 1 - mean_f |z_{club,f} - z_{nat,f}| \)
- Higher values indicate more similar role/performance profile across contexts.

### 1.5 Quadrant thresholds
- Split by medians of club and national performance scores (same labels as `src/ml/consistency.py`):
  - High club + high national: **Elite**
  - High club + low national: **Club Specialist**
  - Low club + high national: **International Specialist**
  - Low club + low national: **Underperformer**

In [ ]:
if USE_PRECOMPUTED_SCORES:
    scores = spark.sql(f"SELECT * FROM {CONSISTENCY_TABLE}").toPandas()
    loadings = spark.sql(f"SELECT component, feature, loading FROM {PCA_LOADINGS}").toPandas()
    club_med = scores["club_performance_score"].median()
    nat_med = scores["national_performance_score"].median()
    weights = (
        loadings.loc[loadings["feature"].isin(FEATURES)]
        .groupby("feature")["loading"]
        .apply(lambda s: s.abs().sum())
        .reindex(FEATURES)
    )
    weights = weights / weights.sum()
    weights_df = pd.DataFrame({"feature": weights.index, "weight": weights.values})
    weights_df["weight_pct"] = (weights_df["weight"] * 100).round(2)
    weights_df = weights_df.sort_values("weight", ascending=False).reset_index(drop=True)
    print(f"Loaded {len(scores):,} precomputed scores from {CONSISTENCY_TABLE}")
    display(weights_df.head())
else:
    players = spark.sql(f"SELECT * FROM {INT_TABLE}").toPandas()
    loadings = spark.sql(f"SELECT component, feature, loading FROM {PCA_LOADINGS}").toPandas()

    missing = set(FEATURES) - set(loadings["feature"])
    if missing:
        raise ValueError(f"Missing in pca_loadings: {missing}")

    weights = (
        loadings.groupby("feature")["loading"]
        .apply(lambda s: s.abs().sum())
        .reindex(FEATURES)
    )
    if weights.isna().any():
        raise ValueError("pca_loadings missing weights for one or more required features.")
    weights = weights / weights.sum()
    weights_df = pd.DataFrame({"feature": weights.index, "weight": weights.values})
    weights_df["weight_pct"] = (weights_df["weight"] * 100).round(2)
    weights_df = weights_df.sort_values("weight", ascending=False).reset_index(drop=True)

    print(f"Rows loaded: {len(players):,}")
    print(f"Distinct players: {players['player_id'].nunique():,}")
    print(f"Weight sum: {weights.sum():.6f}")
    display(weights_df)

---
## 2. Compute scores and quadrants

In [ ]:
if not USE_PRECOMPUTED_SCORES:
    def normalize_bool(val):
        if val is None or (isinstance(val, float) and np.isnan(val)):
            return np.nan
        if isinstance(val, (bool, np.bool_)):
            return bool(val)
        if isinstance(val, (int, float)):
            return bool(val)
        s = str(val).strip().lower()
        if s in ("true", "1", "t", "yes", "national"):
            return True
        if s in ("false", "0", "f", "no", "club"):
            return False
        return np.nan

    players = players.copy()
    players["is_international"] = players["is_international"].apply(normalize_bool)
    players = players.dropna(subset=["is_international"])
    players["is_international"] = players["is_international"].astype(bool)
    players = players.loc[players["total_minutes"] >= MIN_MINUTES].copy()

    w = weights.to_dict()
    players["performance_score"] = players.apply(
        lambda r: float(np.nansum([r[f"z_{f}"] * w[f] for f in FEATURES])), axis=1
    )

    keep_cols = ["player_id", "player_name", "is_international", "total_minutes", "performance_score"] + Z_COLS
    tmp = players[keep_cols].copy()
    tmp = tmp.groupby(["player_id", "is_international"], as_index=False).agg(
        {
            "player_name": "first",
            "total_minutes": "first",
            "performance_score": "first",
            **{c: "first" for c in Z_COLS},
        }
    )

    club = tmp.loc[~tmp["is_international"], ["player_id", "player_name", "performance_score", "total_minutes"]].rename(
        columns={"performance_score": "club_performance_score", "total_minutes": "club_minutes"}
    )
    nat = tmp.loc[tmp["is_international"], ["player_id", "performance_score", "total_minutes"]].rename(
        columns={"performance_score": "national_performance_score", "total_minutes": "national_minutes"}
    )
    scores = club.merge(nat, on="player_id", how="inner", validate="one_to_one")

    club_z = tmp.loc[~tmp["is_international"]].set_index("player_id")[Z_COLS]
    nat_z = tmp.loc[tmp["is_international"]].set_index("player_id")[Z_COLS]
    common_ids = club_z.index.intersection(nat_z.index)
    scores["consistency_score"] = scores["player_id"].map(
        1.0 - (club_z.loc[common_ids] - nat_z.loc[common_ids]).abs().mean(axis=1)
    )

    club_med = scores["club_performance_score"].median()
    nat_med = scores["national_performance_score"].median()

    high_club = scores["club_performance_score"] >= club_med
    high_nat = scores["national_performance_score"] >= nat_med
    scores["performance_quadrant"] = np.select(
        [high_club & high_nat, high_club & ~high_nat, ~high_club & high_nat],
        ["Elite", "Club Specialist", "International Specialist"],
        default="Underperformer",
    )

print(f"Dual-context players scored: {len(scores):,}")
print(f"Median thresholds: club={club_med:.3f}, national={nat_med:.3f}")
display(scores.head(10))

---
## 3. Quadrant distribution + scatter (required)

In [ ]:
quad_counts = scores["performance_quadrant"].value_counts().rename("count")
quad_pct = (quad_counts / quad_counts.sum() * 100).round(1).rename("pct")
quad_table = pd.concat([quad_counts, quad_pct], axis=1).reset_index().rename(columns={"index": "performance_quadrant"})

print("Quadrant distribution:")
display(quad_table)

palette = {
    "Elite": "#2ca02c",
    "Club Specialist": "#1f77b4",
    "International Specialist": "#ff7f0e",
    "Underperformer": "#7f7f7f",
}
fig, ax = plt.subplots(figsize=(9, 7))
for label, grp in scores.groupby("performance_quadrant"):
    ax.scatter(
        grp["club_performance_score"],
        grp["national_performance_score"],
        s=35,
        alpha=0.65,
        c=palette.get(label, "#333333"),
        label=label,
    )
ax.axvline(club_med, color="black", ls="--", lw=0.8, alpha=0.6)
ax.axhline(nat_med, color="black", ls="--", lw=0.8, alpha=0.6)
ax.set_title(f"RQ3: Club vs national performance quadrants ({len(scores):,} players)")
ax.set_xlabel("club_performance_score")
ax.set_ylabel("national_performance_score")
ax.legend(title="Quadrant", loc="best")
plt.tight_layout()
plt.show()

In [ ]:
def top_notable_per_quadrant(n=10):
    out = []
    for q, sub in scores.groupby("performance_quadrant"):
        s = sub.copy()
        s["notable_score"] = (
            s["club_performance_score"].abs() + s["national_performance_score"].abs()
        ) / 2
        top = s.nlargest(n, "notable_score")[
            [
                "player_name",
                "performance_quadrant",
                "club_performance_score",
                "national_performance_score",
                "consistency_score",
                "club_minutes",
                "national_minutes",
            ]
        ]
        out.append(top)
    return pd.concat(out, ignore_index=True)

print("Top 10 notable players per quadrant:")
display(top_notable_per_quadrant(10))

---
## 4. Enrich with position and competition labels

In [ ]:
def _sql_strings(values):
    return ", ".join("'" + v.replace("'", "''") + "'" for v in values)

top5_names_sql = _sql_strings(TOP5_RAW_NAMES)
context_sql = f"""
WITH club_match_rows AS (
  SELECT
    pms.player_id,
    pms.position_name,
    m.competition_name,
    pms.minutes_played
  FROM {INT_MATCH} pms
  INNER JOIN {MATCHES} m ON pms.match_id = m.match_id
  WHERE COALESCE(m.is_international, false) = false
    AND m.competition_name IN ({top5_names_sql})
),
position_minutes AS (
  SELECT player_id, position_name, SUM(minutes_played) AS mins
  FROM club_match_rows
  WHERE position_name IS NOT NULL AND position_name NOT IN ('', 'Unknown')
  GROUP BY 1, 2
),
primary_position AS (
  SELECT player_id, position_name AS primary_position
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY mins DESC) AS rn
    FROM position_minutes
  )
  WHERE rn = 1
),
competition_minutes AS (
  SELECT player_id, competition_name, SUM(minutes_played) AS mins
  FROM club_match_rows
  GROUP BY 1, 2
),
primary_competition AS (
  SELECT player_id, competition_name AS primary_competition_name
  FROM (
    SELECT *, ROW_NUMBER() OVER (PARTITION BY player_id ORDER BY mins DESC) AS rn
    FROM competition_minutes
  )
  WHERE rn = 1
)
SELECT
  p.player_id,
  p.primary_position,
  c.primary_competition_name
FROM primary_position p
LEFT JOIN primary_competition c USING (player_id)
"""

context_df = spark.sql(context_sql).toPandas()
context_df["position_group"] = context_df["primary_position"].map(POSITION_MAP)
context_df["competition_label"] = context_df["primary_competition_name"].map(NAME_TO_LABEL)

scores = scores.merge(context_df, on="player_id", how="left")
scores["position_group"] = pd.Categorical(scores["position_group"], categories=POSITION_ORDER, ordered=True)
scores["competition_label"] = pd.Categorical(scores["competition_label"], categories=COMP_LABEL_ORDER, ordered=True)

print(f"Context rows joined: {context_df['player_id'].nunique():,}")
print("Missing mapped position group:", scores["position_group"].isna().sum())
print("Missing competition label:", scores["competition_label"].isna().sum())

---
## 5. Position breakdown (are defenders more consistent than forwards?)

In [ ]:
pos_stats = (
    scores.dropna(subset=["position_group", "consistency_score"])
    .groupby("position_group")
    .agg(
        n_players=("player_id", "nunique"),
        mean_consistency=("consistency_score", "mean"),
        median_consistency=("consistency_score", "median"),
        std_consistency=("consistency_score", "std"),
    )
    .reindex(POSITION_ORDER)
    .round(3)
)
print("Consistency by position group:")
display(pos_stats)

defenders = scores[scores["position_group"].isin(["CB", "FB"])]["consistency_score"].dropna()
forwards = scores[scores["position_group"] == "FW"]["consistency_score"].dropna()
if len(defenders) > 0 and len(forwards) > 0:
    u, p = mannwhitneyu(defenders, forwards, alternative="two-sided")
    print(
        f"Defenders vs forwards (Mann-Whitney): U={u:.1f}, p={p:.3e}; "
        f"mean_def={defenders.mean():.3f}, mean_fwd={forwards.mean():.3f}"
    )

fig, ax = plt.subplots(figsize=(9, 5))
sns.boxplot(
    data=scores.dropna(subset=["position_group", "consistency_score"]),
    x="position_group",
    y="consistency_score",
    order=POSITION_ORDER,
    ax=ax,
)
ax.set_title("Consistency score distribution by position group")
ax.set_xlabel("Position group")
ax.set_ylabel("consistency_score")
plt.tight_layout()
plt.show()

---
## 6. Competition breakdown (La Liga vs EPL etc.)

In [ ]:
comp_stats = (
    scores.dropna(subset=["competition_label", "consistency_score"])
    .groupby("competition_label")
    .agg(
        n_players=("player_id", "nunique"),
        mean_consistency=("consistency_score", "mean"),
        median_consistency=("consistency_score", "median"),
        std_consistency=("consistency_score", "std"),
    )
    .reindex(COMP_LABEL_ORDER)
    .round(3)
)
print("Consistency by primary club competition:")
display(comp_stats)

quad_comp = pd.crosstab(scores["competition_label"], scores["performance_quadrant"])
quad_comp = quad_comp.reindex(index=COMP_LABEL_ORDER, fill_value=0)
quad_comp_pct = quad_comp.div(quad_comp.sum(axis=1), axis=0).mul(100).round(1)
print("Quadrant composition by competition (row %):")
display(quad_comp_pct)

fig, ax = plt.subplots(figsize=(11, 4))
sns.heatmap(
    quad_comp_pct,
    annot=True,
    fmt=".1f",
    cmap="YlOrRd",
    linewidths=0.5,
    cbar_kws={"label": "% of league row"},
    ax=ax,
)
ax.set_title("Quadrant distribution by competition (row %)")
ax.set_xlabel("Performance quadrant")
ax.set_ylabel("Competition")
plt.tight_layout()
plt.show()

---
## 7. Report-ready summary

In [ ]:
report_summary = pd.DataFrame(
    {
        "metric": [
            "dual_context_players",
            "n_pca_features",
            "club_median_threshold",
            "national_median_threshold",
            "mean_consistency_all",
            "mean_consistency_defenders",
            "mean_consistency_forwards",
            "n_top5_competition_labels_nonnull",
        ],
        "value": [
            len(scores),
            len(FEATURES),
            round(club_med, 3),
            round(nat_med, 3),
            round(scores["consistency_score"].mean(), 3),
            round(defenders.mean(), 3) if len(defenders) else np.nan,
            round(forwards.mean(), 3) if len(forwards) else np.nan,
            int(scores["competition_label"].notna().sum()),
        ],
    }
)
report_summary["value"] = report_summary["value"].astype(str)
print("=== RQ3 executive summary ===")
display(report_summary)

print("\nSuggested report bullets:")
print("  • Scores use 13 PCA-aligned features from dbt_intermediate + analytics.pca_loadings.")
print("  • Quadrant thresholds use median splits of club and national performance scores.")
print("  • Position-level consistency table indicates whether defenders are more stable than forwards.")
print("  • Competition-level tables/heatmap show how consistency patterns vary across top-five leagues.")
print("  • Scatter plot visualizes player distribution across the four quadrants.")